## **Seebeck Fully Modulated output**

In [72]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# Define symbolic functions
# --------------------------
def f_cos(x): return np.cos(x)
def f_sin(x): return np.sin(x)
def f_abs(x): return np.abs(x)
def f_x(x):   return x
def f_gaussian(x): return np.exp(-x**2)
def f_tanh(x): return np.tanh(x)
def f_arctan(x): return np.arctan(x)

func_map = {
    "cos": f_cos, "sin": f_sin, "abs": f_abs, "x": f_x,
    "gaussian": f_gaussian, "tanh": f_tanh, "arctan": f_arctan
}

# --------------------------
# Edge functions from table
# --------------------------
edges_x39 = [
    ("cos",0.9860), ("sin",0.8883), ("cos",0.9904), ("abs",0.9519),
    ("abs",0.9419), ("x",0.9464), ("sin",0.9880), ("cos",0.9582),
    ("cos",0.7517), ("abs",0.9604), ("sin",0.9631), ("sin",0.9922),
    ("cos",0.9742), ("sin",0.9916), ("cos",0.8838), ("cos",0.9856)
]
edges_x83 = [
    ("x",0.8660), ("x",0.8896), ("cos",0.9444), ("sin",0.9659),
    ("sin",0.9918), ("cos",0.9922), ("cos",0.9658), ("cos",0.9842),
    ("cos",0.8703), ("cos",0.9800), ("abs",0.7747), ("cos",0.9934),
    ("x",0.9407), ("gaussian",0.9729), ("abs",0.8528), ("cos",0.9660)
]

# Node functions (layer 1 → output)
nodes = {
    11: ("tanh", 0.9521),
    13: ("abs",  0.9843),
}

# --------------------------
# Build surrogate surface
# --------------------------
def surrogate_surface(node_idx, nx=150, span=3.0, weight_by_r2=False):
    funcs_x39 = [func_map[f] for f, r2 in edges_x39]
    w_x39     = [r2 if weight_by_r2 else 1.0 for f, r2 in edges_x39]
    funcs_x83 = [func_map[f] for f, r2 in edges_x83]
    w_x83     = [r2 if weight_by_r2 else 1.0 for f, r2 in edges_x83]

    node_func = func_map[nodes[node_idx][0]]

    xi_vals = np.linspace(-span, span, nx)   # x_39
    xj_vals = np.linspace(-span, span, nx)   # x_83
    X, Y = np.meshgrid(xi_vals, xj_vals, indexing="ij")  # X: (nx,ny), Y: (nx,ny)

    Z = np.zeros_like(X, dtype=float)
    for w, f in zip(w_x39, funcs_x39):
        Z += w * f(X)
    for w, f in zip(w_x83, funcs_x83):
        Z += w * f(Y)

    Z = node_func(Z)
    if weight_by_r2:
        Z = Z / (np.max(np.abs(Z)) + 1e-12)

    return xi_vals, xj_vals, Z  # shapes: (nx,), (nx,), (nx,ny)

# --------------------------
# Plotting (fixed orientation)
# --------------------------
def plot_heatmap_with_contours(Z, xi_vals, xj_vals, fi, fj, out_png,
                               title, cmap="YlOrRd-r", vmin=None, vmax=None):
    """
    Z is (nx, ny) with indexing='ij' (rows map to xi_vals, cols map to xj_vals).
    Use Zp = Z.T so both imshow and contour align in (x, y) physical space.
    """
    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    Zp = Z.T  # transpose once, use for BOTH imshow and contour

    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.gca()

    im = ax.imshow(
        Zp,
        origin="lower",
        extent=[xi_vals.min(), xi_vals.max(), xj_vals.min(), xj_vals.max()],
        aspect="auto",
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation="nearest"
    )
    # contour: Z shape must be (len(y), len(x)) → Zp matches (xj, xi)
    cs = ax.contour(xi_vals, xj_vals, Zp, levels=12, colors="k", linewidths=0.6, linestyles="--", alpha=0.7)
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2g")

    ax.set_xlabel(r"$x_{39}$ (scaled)", fontsize=16)
    ax.set_ylabel(r"$x_{83}$ (scaled)", fontsize=16)
    ax.set_title(title, fontsize=14, weight="bold", pad=8)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(r"Surrogate contribution (arb. units)", rotation=90)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.grid(alpha=0.25, linestyle="--", linewidth=0.4)

    plt.tight_layout()
    plt.savefig(out_png, dpi=900)
    plt.close()
    print(f"[saved] {out_png}")

# --------------------------
# Compute & plot NODE 11 ONLY
# --------------------------
xi_vals, xj_vals, Z11 = surrogate_surface(node_idx=11, nx=150, span=3.0, weight_by_r2=False)

plot_heatmap_with_contours(
    Z11, xi_vals, xj_vals, 39, 83,
    out_png="results/heatmaps_modulated/heatmap_x39_x83_node11000.png",
    title=r"Surrogate map modulated by $h_{11}$",
    cmap="YlOrRd"
)

# --------------------------
# Compute & plot BOTH NODES
# --------------------------
# xi_vals, xj_vals, Z11 = surrogate_surface(node_idx=11, nx=150, span=3.0)
_,        _,        Z13 = surrogate_surface(node_idx=13, nx=150, span=3.0)

# shared colour scale
vmin = min(Z11.min(), Z13.min())
vmax = max(Z11.max(), Z13.max())

# plot_heatmap_with_contours(
#     Z11, xi_vals, xj_vals, 39, 83,
#     out_png="results/heatmaps_modulated/heatmap_x39_x83_node11.png",
#     title=r"Surrogate map modulated by $h_{11}$",
#     cmap="autumn", vmin=vmin, vmax=vmax
# )

plot_heatmap_with_contours(
    Z13, xi_vals, xj_vals, 39, 83,
    out_png="results/heatmaps_modulated/heatmap_x39_x83_node13000.png",
    title=r"Surrogate map modulated by $h_{13}$",
    cmap="YlOrRd", vmin=vmin, vmax=vmax
)

[saved] results/heatmaps_modulated/heatmap_x39_x83_node11000.png
[saved] results/heatmaps_modulated/heatmap_x39_x83_node13000.png


## **Band Gap Fully Modulated output**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# Define symbolic functions
# --------------------------
def f_cos(x): return np.cos(x)
def f_sin(x): return np.sin(x)
def f_abs(x): return np.abs(x)
def f_x(x):   return x
def f_gaussian(x): return np.exp(-x**2)
def f_tanh(x): return np.tanh(x)
def f_arctan(x): return np.arctan(x)

func_map = {
    "cos": f_cos, "sin": f_sin, "abs": f_abs, "x": f_x,
    "gaussian": f_gaussian, "tanh": f_tanh, "arctan": f_arctan
}

# --------------------------
# Edge functions from table
# --------------------------


edges_x39 = [
    ("sin",      0.9907),
    ("sin",      0.8074),
    ("cos",      0.9824),
    ("sin",      0.9841),
    ("cos",      0.8525),
    ("abs",      0.9931),
    ("gaussian", 0.8167),
    ("abs",      0.9664),
    ("cos",      0.9866),
    ("gaussian", 0.9285),
    ("sin",      0.9917),
    ("gaussian", 0.9848),
    ("abs",      0.9938),
    ("gaussian", 0.9896),
    ("cos",      0.9428),
    ("sin",      0.9375),
]

# x68 (Layer 0, In_idx=68): (Function, R2, c)
edges_x83 = [
    ("gaussian", 0.9165),
    ("abs",      0.9297),
    ("abs",      0.9535),
    ("sin",      0.9923),
    ("gaussian", 0.9876),
    ("sin",      0.9894),
    ("cos",      0.9750),
    ("sin",      0.9548),
    ("cos",      0.9648),
    ("sin",      0.9933),
    ("gaussian", 0.9911),
    ("gaussian", 0.9881),
    ("cos",      0.8568),
    ("sin",      0.9856),
    ("cos",      0.9637),
    ("abs",      0.9951),
]

# Node functions (layer 1 → output)
nodes = {
    11: ("sin", 0.9521),
    13: ("cos",  0.9843),
}

# --------------------------
# Build surrogate surface
# --------------------------
def surrogate_surface(node_idx, nx=150, span=3.0, weight_by_r2=False):
    funcs_x39 = [func_map[f] for f, r2 in edges_x39]
    w_x39     = [r2 if weight_by_r2 else 1.0 for f, r2 in edges_x39]
    funcs_x83 = [func_map[f] for f, r2 in edges_x83]
    w_x83     = [r2 if weight_by_r2 else 1.0 for f, r2 in edges_x83]

    node_func = func_map[nodes[node_idx][0]]

    xi_vals = np.linspace(-span, span, nx)   # x_39
    xj_vals = np.linspace(-span, span, nx)   # x_83
    X, Y = np.meshgrid(xi_vals, xj_vals, indexing="ij")  # X: (nx,ny), Y: (nx,ny)

    Z = np.zeros_like(X, dtype=float)
    for w, f in zip(w_x39, funcs_x39):
        Z += w * f(X)
    for w, f in zip(w_x83, funcs_x83):
        Z += w * f(Y)

    Z = node_func(Z)
    if weight_by_r2:
        Z = Z / (np.max(np.abs(Z)) + 1e-12)

    return xi_vals, xj_vals, Z  # shapes: (nx,), (nx,), (nx,ny)

# --------------------------
# Plotting (fixed orientation)
# --------------------------
def plot_heatmap_with_contours(Z, xi_vals, xj_vals, fi, fj, out_png,
                               title, cmap="YlOrRd", vmin=None, vmax=None):
    """
    Z is (nx, ny) with indexing='ij' (rows map to xi_vals, cols map to xj_vals).
    Use Zp = Z.T so both imshow and contour align in (x, y) physical space.
    """
    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    Zp = Z.T  # transpose once, use for BOTH imshow and contour

    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.gca()

    im = ax.imshow(
        Zp,
        origin="lower",
        extent=[xi_vals.min(), xi_vals.max(), xj_vals.min(), xj_vals.max()],
        aspect="auto",
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation="nearest"
    )
    # contour: Z shape must be (len(y), len(x)) → Zp matches (xj, xi)
    cs = ax.contour(xi_vals, xj_vals, Zp, levels=12, colors="k", linewidths=0.6, linestyles="--", alpha=0.7)
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2g")

    ax.set_xlabel(r"$x_{39}$ (scaled)", fontsize=16)
    ax.set_ylabel(r"$x_{83}$ (scaled)", fontsize=16)
    ax.set_title(title, fontsize=14, weight="bold", pad=8)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(r"Surrogate contribution (arb. units)", rotation=90)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.grid(alpha=0.25, linestyle="--", linewidth=0.4)

    plt.tight_layout()
    plt.savefig(out_png, dpi=900)
    plt.close()
    print(f"[saved] {out_png}")

# --------------------------
# Compute & plot NODE 11 ONLY
# --------------------------
xi_vals, xj_vals, Z11 = surrogate_surface(node_idx=11, nx=150, span=3.0, weight_by_r2=False)

plot_heatmap_with_contours(
    Z11, xi_vals, xj_vals, 39, 83,
    out_png="results/bandgap_heatmaps/heatmap_x39_x83_node11000.png",
    title=r"Surrogate map modulated by $h_{11}$",
    cmap="YlGnBu"
)

# --------------------------
# Compute & plot BOTH NODES
# --------------------------
# xi_vals, xj_vals, Z11 = surrogate_surface(node_idx=11, nx=150, span=3.0)
_,        _,        Z13 = surrogate_surface(node_idx=13, nx=150, span=3.0)

# shared colour scale
vmin = min(Z11.min(), Z13.min())
vmax = max(Z11.max(), Z13.max())

# plot_heatmap_with_contours(
#     Z11, xi_vals, xj_vals, 39, 83,
#     out_png="results/heatmaps_modulated/heatmap_x39_x83_node11.png",
#     title=r"Surrogate map modulated by $h_{11}$",
#     cmap="autumn", vmin=vmin, vmax=vmax
# )

plot_heatmap_with_contours(
    Z13, xi_vals, xj_vals, 39, 83,
    out_png="results/bandgap_heatmaps/heatmap_x39_x83_node13000.png",
    title=r"Surrogate map modulated by $h_{13}$",
    cmap="YlGnBu", vmin=vmin, vmax=vmax
)

[saved] results/bandgap_heatmaps/heatmap_x39_x83_node11000.png
[saved] results/bandgap_heatmaps/heatmap_x39_x83_node13000.png


## **Seebeck simplified output**  

In [79]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ======================
# Surrogate definitions
# ======================
def s11(x39, x83):
    return np.sin(x39) + np.cos(x83)

def h11(x39, x83):
    return np.tanh(s11(x39, x83))

def s13(x39, x83):
    return np.sin(x39) + np.exp(-x83**2)

def h13(x39, x83):
    return np.abs(s13(x39, x83))

def y_simple(x39, x83):
    return h11(x39, x83) + h13(x39, x83)

# ======================
# Grid + plotting utils
# ======================
def make_grid(span=3.0, n=201):
    xi = np.linspace(-span, span, n)   # x_39 (scaled)
    xj = np.linspace(-span, span, n)   # x_83 (scaled)
    X, Y = np.meshgrid(xi, xj, indexing="ij")  # X[i,j]=xi[i], Y[i,j]=xj[j]
    return xi, xj, X, Y

def plot_heatmap_with_contours(Z, xi, xj, out_png, title,
                               cmap="autumn", vmin=None, vmax=None):
    """Consistent orientation: use Z.T for both imshow and contour."""
    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    Zp = Z.T  # align orientation for both imshow and contour

    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.gca()

    im = ax.imshow(
        Zp, origin="lower",
        extent=[xi.min(), xi.max(), xj.min(), xj.max()],
        aspect="auto", cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation="nearest"
    )
    cs = ax.contour(xi, xj, Zp, levels=14, colors="k", linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2g")

    ax.set_xlabel(r"$x_{39}$ (scaled)", fontsize=16)
    ax.set_ylabel(r"$x_{83}$ (scaled)", fontsize=16)
    ax.set_title(title, fontsize=16, weight="bold", pad=8)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(title, rotation=90)

    for s in ("top","right"):
        ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25, linestyle="--", linewidth=0.4)

    plt.tight_layout()
    plt.savefig(out_png, dpi=900)
    plt.close()
    print(f"[saved] {out_png}")

# ======================
# Build and plot all 5
# ======================
xi, xj, X, Y = make_grid(span=3.0, n=201)

# Pre-activation maps
Z_s11 = s11(X, Y)
Z_s13 = s13(X, Y)
# Use a shared color scale for the two pre-activations
vmin_pre = min(Z_s11.min(), Z_s13.min())
vmax_pre = max(Z_s11.max(), Z_s13.max())

plot_heatmap_with_contours(
    Z_s11, xi, xj,
    out_png="results/heatmaps_simple/s11_x39_x83.png",
    title=r"$s_{11}(x_{39},x_{83})$",
    cmap="YlOrRd", vmin=vmin_pre, vmax=vmax_pre
)
plot_heatmap_with_contours(
    Z_s13, xi, xj,
    out_png="results/heatmaps_simple/s13_x39_x83.png",
    title=r"$s_{13}(x_{39},x_{83})$",
    cmap="YlOrRd", vmin=vmin_pre, vmax=vmax_pre
)

# Post-activation maps
Z_h11 = h11(X, Y)
Z_h13 = h13(X, Y)
# Shared color scale for the two post-activations
vmin_post = min(Z_h11.min(), Z_h13.min())
vmax_post = max(Z_h11.max(), Z_h13.max())

plot_heatmap_with_contours(
    Z_h11, xi, xj,
    out_png="results/heatmaps_simple/h11_x39_x83.png",
    title=r"$h_{11}(x_{39},x_{83})$",
    cmap="YlOrRd", vmin=vmin_post, vmax=vmax_post
)
plot_heatmap_with_contours(
    Z_h13, xi, xj,
    out_png="results/heatmaps_simple/h13_x39_x83.png",
    title=r"$h_{13}(x_{39},x_{83})$",
    cmap="YlOrRd", vmin=vmin_post, vmax=vmax_post
)

# Combined surrogate output
Z_y = y_simple(X, Y)
plot_heatmap_with_contours(
    Z_y, xi, xj,
    out_png="results/heatmaps_simple/y_simple_x39_x83.png",
    title=r"$y_{\mathrm{simple}}(x_{39},x_{83})$",
    cmap="YlOrRd"  # own scale (bounded by construction)
)

[saved] results/heatmaps_simple/s11_x39_x83.png
[saved] results/heatmaps_simple/s13_x39_x83.png
[saved] results/heatmaps_simple/h11_x39_x83.png
[saved] results/heatmaps_simple/h13_x39_x83.png
[saved] results/heatmaps_simple/y_simple_x39_x83.png


## **Band Gap simplified output**

In [74]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ======================
# Surrogate definitions
# ======================
def s9(x39, x68):
    return np.exp(-x39**2) + np.sin(x68)

def h9(x39, x68):
    return np.cos(s9(x39, x68))

def s13(x39, x68):
    return np.exp(-x39**2) + np.sin(x68)

def h13(x39, x68):
    return np.sin(s13(x39, x68))

def y_simple(x39, x68):
    return h11(x39, x68) + h13(x39, x68)

# ======================
# Grid + plotting utils
# ======================
def make_grid(span=3.0, n=201):
    xi = np.linspace(-span, span, n)   # x_39 (scaled)
    xj = np.linspace(-span, span, n)   # x_83 (scaled)
    X, Y = np.meshgrid(xi, xj, indexing="ij")  # X[i,j]=xi[i], Y[i,j]=xj[j]
    return xi, xj, X, Y

def plot_heatmap_with_contours(Z, xi, xj, out_png, title,
                               cmap="YlGnBu", vmin=None, vmax=None):
    """Consistent orientation: use Z.T for both imshow and contour."""
    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    Zp = Z.T  # align orientation for both imshow and contour

    fig = plt.figure(figsize=(7.2, 6.2))
    ax = plt.gca()

    im = ax.imshow(
        Zp, origin="lower",
        extent=[xi.min(), xi.max(), xj.min(), xj.max()],
        aspect="auto", cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation="nearest"
    )
    cs = ax.contour(xi, xj, Zp, levels=14, colors="k", linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2g")

    ax.set_xlabel(r"$x_{39}$ (scaled)", fontsize=16)
    ax.set_ylabel(r"$x_{83}$ (scaled)", fontsize=16)
    ax.set_title(title, fontsize=16, weight="bold", pad=8)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(title, rotation=90)

    for s in ("top","right"):
        ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25, linestyle="--", linewidth=0.4)

    plt.tight_layout()
    plt.savefig(out_png, dpi=900)
    plt.close()
    print(f"[saved] {out_png}")

# ======================
# Build and plot all 5
# ======================
xi, xj, X, Y = make_grid(span=3.0, n=201)

# Pre-activation maps
Z_s11 = s11(X, Y)
Z_s13 = s13(X, Y)
# Use a shared color scale for the two pre-activations
vmin_pre = min(Z_s11.min(), Z_s13.min())
vmax_pre = max(Z_s11.max(), Z_s13.max())

plot_heatmap_with_contours(
    Z_s11, xi, xj,
    out_png="results/heatmaps_simple_band_gap/s11_x39_x68.png",
    title=r"$s_{11}(x_{39},x_{68})$",
    cmap="YlGnBu", vmin=vmin_pre, vmax=vmax_pre
)
plot_heatmap_with_contours(
    Z_s13, xi, xj,
    out_png="results/heatmaps_simple_band_gap/s13_x39_x68.png",
    title=r"$s_{13}(x_{39},x_{68})$",
    cmap="YlGnBu", vmin=vmin_pre, vmax=vmax_pre
)

# Post-activation maps
Z_h11 = h11(X, Y)
Z_h13 = h13(X, Y)
# Shared color scale for the two post-activations
vmin_post = min(Z_h11.min(), Z_h13.min())
vmax_post = max(Z_h11.max(), Z_h13.max())

plot_heatmap_with_contours(
    Z_h11, xi, xj,
    out_png="results/heatmaps_simple_band_gap/h11_x39_x68.png",
    title=r"$h_{11}(x_{39},x_{68})$",
    cmap="YlGnBu", vmin=vmin_post, vmax=vmax_post
)
plot_heatmap_with_contours(
    Z_h13, xi, xj,
    out_png="results/heatmaps_simple_band_gap/h13_x39_x68.png",
    title=r"$h_{13}(x_{39},x_{68})$",
    cmap="YlGnBu", vmin=vmin_post, vmax=vmax_post
)

# Combined surrogate output
Z_y = y_simple(X, Y)
plot_heatmap_with_contours(
    Z_y, xi, xj,
    out_png="results/heatmaps_simple_band_gap/y_simple_x39_x68.png",
    title=r"$y_{\mathrm{simple}}(x_{39},x_{68})$",
    cmap="YlGnBu"  # own scale (bounded by construction)
)

[saved] results/heatmaps_simple_band_gap/s11_x39_x68.png
[saved] results/heatmaps_simple_band_gap/s13_x39_x68.png
[saved] results/heatmaps_simple_band_gap/h11_x39_x68.png
[saved] results/heatmaps_simple_band_gap/h13_x39_x68.png
[saved] results/heatmaps_simple_band_gap/y_simple_x39_x68.png
